# Silver → Gold · fact_ventas_revendedor (hecho)
Generado con la skill `notebooks-medallion` desde `metadata/silver_gold/fact_ventas_revendedor.json` y `fact_ventas_revendedor.sql`.
**No lo edites a mano**: cambia la consulta o la metadata y vuelve a generar.

Ventas a revendedores (OnlineOrderFlag = 0) a grano de línea de pedido. Portes e impuestos de cabecera repartidos por el peso del importe de la línea; el resto de redondeo va a la última línea, así que por pedido cuadran al céntimo con la cabecera.

- **Destino:** `gold.dbo.fact_ventas_revendedor`. **Grano y clave del MERGE:** `IdPedido, IdLineaPedido`.
- **Lee:** `silver.Sales.SalesOrderDetail`, `silver.Sales.SalesOrderHeader`, `silver.Production.Product`, `silver.Production.ProductCostHistory`, `silver.Sales.Customer`.
- Inserta, actualiza (solo si cambia la fila) y borra lo que ya no devuelve la consulta. Sin SCD2.

In [ ]:
proyecto = "GreatAdventures"
ws_datos = "GreatAdventures"
ws_control = "GreatAdventuresControl"
lh_silver = "silver"
lh_gold = "gold"
lh_control = "control"
entidad = "fact_ventas_revendedor"
esquema_destino = "dbo"
tabla_destino = "fact_ventas_revendedor"
borrar_ausentes = True
evolucion_esquema = True   # añadir a Gold las columnas nuevas de la consulta
simular = False            # True: ejecuta la consulta, valida y registra en el log, pero no escribe en Gold
id_orquestacion = ""       # lo rellena el orquestador

In [ ]:
# Utilidades comunes (iguales en todos los notebooks generados): rutas, log en el lakehouse de control y MERGE.
import json, uuid, datetime as dt
from delta.tables import DeltaTable
from pyspark.sql import functions as F, types as T, Window

ONELAKE = "onelake.dfs.fabric.microsoft.com"

def ruta_lakehouse(ws, lh):
    return f"abfss://{ws}@{ONELAKE}/{lh}.Lakehouse"

RUTA_CONTROL = f"{ruta_lakehouse(ws_control, lh_control)}/Tables/dbo"
ID_EJECUCION = str(uuid.uuid4())
INICIO = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)

def contexto(clave, defecto=None):
    try:
        return notebookutils.runtime.context.get(clave) or defecto
    except Exception:
        return defecto

ESQUEMA_LOG_EJECUCION = T.StructType([
    T.StructField("id_ejecucion", T.StringType()),
    T.StructField("id_orquestacion", T.StringType()),
    T.StructField("proyecto", T.StringType()),
    T.StructField("capa", T.StringType()),
    T.StructField("objeto", T.StringType()),
    T.StructField("destino", T.StringType()),
    T.StructField("tipo_carga", T.StringType()),
    T.StructField("estado", T.StringType()),
    T.StructField("simulado", T.BooleanType()),
    T.StructField("inicio", T.TimestampType()),
    T.StructField("fin", T.TimestampType()),
    T.StructField("duracion_s", T.DoubleType()),
    T.StructField("ficheros_leidos", T.LongType()),
    T.StructField("filas_leidas", T.LongType()),
    T.StructField("filas_unicas", T.LongType()),
    T.StructField("filas_duplicadas", T.LongType()),
    T.StructField("filas_borrado_origen", T.LongType()),
    T.StructField("filas_insertadas", T.LongType()),
    T.StructField("filas_actualizadas", T.LongType()),
    T.StructField("filas_borradas", T.LongType()),
    T.StructField("marca_desde", T.StringType()),
    T.StructField("marca_hasta", T.StringType()),
    T.StructField("error", T.StringType()),
    T.StructField("notebook", T.StringType()),
    T.StructField("workspace", T.StringType()),
    T.StructField("usuario", T.StringType()),
    T.StructField("spark_app_id", T.StringType()),
    T.StructField("parametros", T.StringType()),
])

def registrar_ejecucion(**valores):
    """Añade una fila a control.dbo.log_ejecucion. Solo append: no hay conflictos entre notebooks en paralelo."""
    fin = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
    base = dict(id_ejecucion=ID_EJECUCION, proyecto=proyecto, inicio=INICIO, fin=fin,
                duracion_s=(fin - INICIO).total_seconds(), notebook=contexto("currentNotebookName"),
                workspace=contexto("currentWorkspaceName"), usuario=contexto("userName"),
                spark_app_id=spark.sparkContext.applicationId)
    base.update(valores)
    fila = tuple(base.get(c.name) for c in ESQUEMA_LOG_EJECUCION)
    (spark.createDataFrame([fila], ESQUEMA_LOG_EJECUCION)
         .write.format("delta").mode("append").option("mergeSchema", "true")
         .save(f"{RUTA_CONTROL}/log_ejecucion"))

def es_tabla_delta(ruta):
    try:
        return DeltaTable.isDeltaTable(spark, ruta)
    except Exception:
        return False

def con_hash(df, excluir):
    """Añade _hash_fila: sha2 de las columnas de negocio. Sirve para no reescribir filas que no han cambiado."""
    cols = [c for c in df.columns if c not in excluir and not c.startswith("_")]
    return df.withColumn("_hash_fila", F.sha2(F.concat_ws("\u0001", *[F.coalesce(F.col(c).cast("string"), F.lit("\u0000")) for c in sorted(cols)]), 256))

def anadir_columnas_nuevas(ruta, df, evolucion):
    """Evolución de esquema controlada: añade al destino las columnas nuevas del origen o falla si no está permitido."""
    destino = spark.read.format("delta").load(ruta)
    nuevas = [f for f in df.schema.fields if f.name not in destino.columns]
    if not nuevas:
        return []
    if not evolucion:
        raise ValueError(f"Columnas nuevas en origen y evolucion_esquema=False: {[f.name for f in nuevas]}")
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in nuevas)
    spark.sql(f"ALTER TABLE delta.`{ruta}` ADD COLUMNS ({cols})")
    return [f.name for f in nuevas]

def columnas_con_tipo_cambiado(ruta, df):
    """Columnas que existen en destino con un tipo distinto al del origen."""
    actual = {f.name: f.dataType for f in spark.read.format("delta").load(ruta).schema.fields}
    return [f"{f.name}: {actual[f.name].simpleString()} -> {f.dataType.simpleString()}"
            for f in df.schema.fields if f.name in actual and actual[f.name] != f.dataType]

def fusionar(ruta, df, claves, borrar_ausentes, col_borrado=None, cambio_tipo="ignorar"):
    """MERGE contra Delta. df ya viene deduplicado por claves y con _hash_fila.
    - col_borrado (opcional): columna booleana del origen; si es True, la fila se borra en destino.
    - borrar_ausentes: borra del destino lo que no viene en el origen (instantánea completa).
    - cambio_tipo: qué hacer si una columna existente cambia de tipo en el origen.
        "ignorar"    el MERGE convierte al tipo del destino (comportamiento histórico).
        "reescribir" reescribe la tabla con el esquema del origen (solo tiene sentido con borrar_ausentes).
        "fallar"     lanza un error.
    Devuelve (insertadas, actualizadas, borradas)."""
    cols = [c for c in df.columns if c != col_borrado]
    if not es_tabla_delta(ruta):
        datos = df.filter(~F.col(col_borrado)) if col_borrado else df
        datos.select(*cols).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ruta)
        return spark.read.format("delta").load(ruta).count(), 0, 0
    if cambio_tipo != "ignorar":
        cambiados = columnas_con_tipo_cambiado(ruta, df.select(*cols))
        if cambiados and cambio_tipo == "fallar":
            raise ValueError(f"Cambio de tipo en {cambiados} y la tabla no se puede reescribir (borrar_ausentes=False)")
        if cambiados:
            print(f"Cambio de tipo en {cambiados}: se reescribe la tabla con el esquema de la consulta")
            antes = spark.read.format("delta").load(ruta).count()
            datos = df.filter(~F.col(col_borrado)) if col_borrado else df
            datos.select(*cols).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ruta)
            return spark.read.format("delta").load(ruta).count(), 0, antes
    anadir_columnas_nuevas(ruta, df.select(*cols), evolucion_esquema)
    cond = " AND ".join(f"t.`{c}` <=> s.`{c}`" for c in claves)
    asignar = {f"`{c}`": f"s.`{c}`" for c in cols}
    m = DeltaTable.forPath(spark, ruta).alias("t").merge(df.alias("s"), cond)
    if col_borrado:
        m = m.whenMatchedDelete(condition=f"s.`{col_borrado}`")
        m = m.whenMatchedUpdate(condition=f"NOT s.`{col_borrado}` AND NOT (t._hash_fila <=> s._hash_fila)", set=asignar)
        m = m.whenNotMatchedInsert(condition=f"NOT s.`{col_borrado}`", values=asignar)
    else:
        m = m.whenMatchedUpdate(condition="NOT (t._hash_fila <=> s._hash_fila)", set=asignar)
        m = m.whenNotMatchedInsert(values=asignar)
    if borrar_ausentes:
        m = m.whenNotMatchedBySourceDelete()
    m.execute()
    met = DeltaTable.forPath(spark, ruta).history(1).select("operationMetrics").first()[0] or {}
    return (int(met.get("numTargetRowsInserted", 0)), int(met.get("numTargetRowsUpdated", 0)),
            int(met.get("numTargetRowsDeleted", 0)))

In [ ]:
CAPA = "silver_gold"
CLAVE = ["IdPedido", "IdLineaPedido"]
RUTA_GOLD = f"{ruta_lakehouse(ws_datos, lh_gold)}/Tables/{esquema_destino}/{tabla_destino}"
LAKEHOUSES = {"silver": lh_silver, "gold": lh_gold}
# Tablas que lee la consulta: se registran como vistas temporales <capa>_<esquema>_<tabla>.
DEPENDENCIAS = [{"capa": "silver", "esquema": "Sales", "tabla": "SalesOrderDetail", "vista": "silver_Sales_SalesOrderDetail"}, {"capa": "silver", "esquema": "Sales", "tabla": "SalesOrderHeader", "vista": "silver_Sales_SalesOrderHeader"}, {"capa": "silver", "esquema": "Production", "tabla": "Product", "vista": "silver_Production_Product"}, {"capa": "silver", "esquema": "Production", "tabla": "ProductCostHistory", "vista": "silver_Production_ProductCostHistory"}, {"capa": "silver", "esquema": "Sales", "tabla": "Customer", "vista": "silver_Sales_Customer"}]

CONSULTA = r"""
-- fact_ventas_revendedor · grano: línea de pedido de revendedor (IdPedido, IdLineaPedido)
-- Canal: OnlineOrderFlag = 0 (revendedor). Portes e impuestos son de CABECERA: se reparten por el peso del
-- importe de la línea y el resto de redondeo va a la última línea, para que por pedido sumen exactamente lo de la cabecera.
-- Coste: el coste estándar vigente en la fecha del pedido (Production.ProductCostHistory); si no hay, el de Production.Product.
-- Claves Id* a BIGINT: en silver StoreID y SalesPersonID son double porque admiten nulos.
WITH coste AS (
    SELECT d.SalesOrderDetailID,
           CAST(COALESCE(ch.StandardCost, p.StandardCost) AS DECIMAL(19, 4)) AS CosteUnitario,
           ROW_NUMBER() OVER (PARTITION BY d.SalesOrderDetailID ORDER BY ch.StartDate DESC) AS n
    FROM silver_Sales_SalesOrderDetail d
    JOIN silver_Sales_SalesOrderHeader h      ON h.SalesOrderID = d.SalesOrderID
    JOIN silver_Production_Product p          ON p.ProductID = d.ProductID
    LEFT JOIN silver_Production_ProductCostHistory ch
           ON ch.ProductID = d.ProductID
          AND CAST(ch.StartDate AS DATE) <= CAST(h.OrderDate AS DATE)
          AND (ch.EndDate IS NULL OR CAST(h.OrderDate AS DATE) <= CAST(ch.EndDate AS DATE))
    WHERE CAST(h.OnlineOrderFlag AS INT) = 0
),
lineas AS (
    SELECT
        h.SalesOrderID, d.SalesOrderDetailID, h.SalesOrderNumber,
        CAST(h.OrderDate AS DATE) AS FechaPedido, CAST(h.DueDate AS DATE) AS FechaVencimiento,
        CAST(h.ShipDate AS DATE) AS FechaEnvio,
        d.ProductID, CAST(c.StoreID AS BIGINT) AS StoreID, h.TerritoryID,
        CAST(h.SalesPersonID AS BIGINT) AS SalesPersonID, d.SpecialOfferID,
        d.OrderQty,
        CAST(d.UnitPrice AS DECIMAL(19, 4))         AS PrecioUnitario,
        CAST(d.UnitPriceDiscount AS DECIMAL(19, 4)) AS DescuentoUnitario,
        CAST(d.LineTotal AS DECIMAL(19, 4))         AS ImporteLinea,
        CAST(d.UnitPrice * d.OrderQty AS DECIMAL(19, 4)) AS ImporteBruto,
        co.CosteUnitario,
        CAST(h.Freight AS DECIMAL(19, 4))           AS PortesPedido,
        CAST(h.TaxAmt AS DECIMAL(19, 4))            AS ImpuestosPedido,
        SUM(CAST(d.LineTotal AS DECIMAL(19, 4))) OVER (PARTITION BY h.SalesOrderID) AS ImportePedido,
        COUNT(*) OVER (PARTITION BY h.SalesOrderID)                                AS LineasPedido,
        ROW_NUMBER() OVER (PARTITION BY h.SalesOrderID ORDER BY d.SalesOrderDetailID DESC) AS OrdenInverso
    FROM silver_Sales_SalesOrderHeader h
    JOIN silver_Sales_SalesOrderDetail d ON d.SalesOrderID = h.SalesOrderID
    JOIN silver_Sales_Customer c         ON c.CustomerID = h.CustomerID
    JOIN coste co                        ON co.SalesOrderDetailID = d.SalesOrderDetailID AND co.n = 1
    WHERE CAST(h.OnlineOrderFlag AS INT) = 0
),
reparto AS (
    SELECT *,
        CAST(ROUND(CASE WHEN ImportePedido = 0 THEN PortesPedido / LineasPedido
                        ELSE PortesPedido * ImporteLinea / ImportePedido END, 4) AS DECIMAL(19, 4))    AS PortesProv,
        CAST(ROUND(CASE WHEN ImportePedido = 0 THEN ImpuestosPedido / LineasPedido
                        ELSE ImpuestosPedido * ImporteLinea / ImportePedido END, 4) AS DECIMAL(19, 4)) AS ImpuestosProv
    FROM lineas
)
SELECT
    SalesOrderID                AS IdPedido,
    SalesOrderDetailID          AS IdLineaPedido,
    SalesOrderNumber            AS NumeroPedido,
    FechaPedido, FechaVencimiento, FechaEnvio,
    ProductID                   AS IdProducto,
    StoreID                     AS IdRevendedor,
    TerritoryID                 AS IdTerritorio,
    SalesPersonID               AS IdVendedor,
    SpecialOfferID              AS IdPromocion,
    OrderQty                    AS Cantidad,
    PrecioUnitario, DescuentoUnitario, ImporteBruto, ImporteLinea,
    CosteUnitario,
    CAST(CosteUnitario * OrderQty AS DECIMAL(19, 4))        AS Coste,
    CAST(CASE WHEN OrdenInverso = 1
              THEN PortesPedido - (SUM(PortesProv) OVER (PARTITION BY SalesOrderID) - PortesProv)
              ELSE PortesProv END AS DECIMAL(19, 4))        AS Portes,
    CAST(CASE WHEN OrdenInverso = 1
              THEN ImpuestosPedido - (SUM(ImpuestosProv) OVER (PARTITION BY SalesOrderID) - ImpuestosProv)
              ELSE ImpuestosProv END AS DECIMAL(19, 4))     AS Impuestos
FROM reparto
"""

In [ ]:
m = dict(ficheros_leidos=0, filas_leidas=0, filas_unicas=0, filas_duplicadas=0, filas_borrado_origen=0,
         filas_insertadas=0, filas_actualizadas=0, filas_borradas=0)
estado, error = "OK", None
try:
    for d in DEPENDENCIAS:
        ruta = f"{ruta_lakehouse(ws_datos, LAKEHOUSES[d['capa']])}/Tables/{d['esquema']}/{d['tabla']}"
        spark.read.format("delta").load(ruta).createOrReplaceTempView(d["vista"])

    df = spark.sql(CONSULTA).cache()
    faltan = [c for c in CLAVE if c not in df.columns]
    if faltan:
        raise ValueError(f"{entidad}: la consulta no devuelve las columnas de la clave {faltan}")
    m["filas_leidas"] = df.count()
    nulos = df.filter(" OR ".join(f"`{c}` IS NULL" for c in CLAVE)).count()
    if nulos:
        raise ValueError(f"{entidad}: {nulos} filas con la clave {CLAVE} a NULL")
    m["filas_unicas"] = df.select(*CLAVE).distinct().count()
    m["filas_duplicadas"] = m["filas_leidas"] - m["filas_unicas"]
    if m["filas_duplicadas"]:
        raise ValueError(f"{entidad}: la consulta rompe el grano declarado {CLAVE}: {m['filas_duplicadas']} filas duplicadas")

    salida = (con_hash(df, set())
              .withColumn("_id_ejecucion", F.lit(ID_EJECUCION))
              .withColumn("_procesado", F.current_timestamp()))
    if not simular:
        # Gold es la foto de la consulta: si cambia un tipo, se reescribe la tabla (o falla si no se puede borrar).
        ins, act, bor = fusionar(RUTA_GOLD, salida, CLAVE, borrar_ausentes=borrar_ausentes,
                                 cambio_tipo="reescribir" if borrar_ausentes else "fallar")
        m.update(filas_insertadas=ins, filas_actualizadas=act, filas_borradas=bor)
    df.unpersist()
except Exception as e:
    estado, error = "ERROR", f"{type(e).__name__}: {e}"[:4000]
finally:
    registrar_ejecucion(id_orquestacion=id_orquestacion or None, capa=CAPA, objeto=entidad,
                        destino=f"{lh_gold}.{esquema_destino}.{tabla_destino}", tipo_carga="merge",
                        estado=estado, simulado=bool(simular), error=error,
                        parametros=json.dumps(dict(borrar_ausentes=borrar_ausentes, simular=simular)), **m)
if error:
    raise RuntimeError(error)

In [ ]:
notebookutils.notebook.exit(json.dumps(dict(estado=estado, entidad=entidad, id_ejecucion=ID_EJECUCION, **m)))